# SPR-03 — Model Training + Ensembling

**Ticket:** SPR-03
**Owner:** Sakhiur
**Depends on:** SPR-02 (resampled train set)
**Folder:** `notebooks/04_modeling/02_ensembling.ipynb`
**Output:** trained models saved to `models/{decision_tree,logistic_regression,random_forest,xgboost}/`, metrics logged to `results/tables/`

**Inspiration:** `SOTA_Paper_8.ipynb` cells 37-48 (hyperparameter search + VotingClassifier hard/soft voting), `cross-validation.ipynb` cells 37/40 (Stratified K-Fold evaluation loop). Adapted for multi-class `income_class` and depth-capped models to avoid the Colab stall documented earlier in the project.

## 1. Load resampled train set from SPR-02

In [1]:
!wget https://huggingface.co/datasets/Sakhiur/signal/resolve/main/X_train_resampled_Sakhiur.csv

--2026-08-25 03:45:16--  https://huggingface.co/datasets/Sakhiur/signal/resolve/main/X_train_resampled_Sakhiur.csv
Resolving huggingface.co (huggingface.co)... 3.169.137.19, 3.169.137.111, 3.169.137.5, ...
Connecting to huggingface.co (huggingface.co)|3.169.137.19|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/6a7c7183f2c81824bdfb6fa9/9f6172a1773ccf26810ab1921082ea2b55dd1e0849d93107fe3e14ac836e8fd0?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27X_train_resampled_Sakhiur.csv%3B+filename%3D%22X_train_resampled_Sakhiur.csv%22%3B&response-content-type=text%2Fcsv&user_id=public&X-Xet-Cas-Uid=public&Expires=1787633116&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNmE3YzcxODNmMmM4MTgyNGJkZmI2ZmE5LzlmNjE3MmExNzczY2NmMjY4MTBhYjE5MjEwODJlYTJiNTVkZDFlMDg0OWQ5MzEwN2ZlM2UxNGFjODM2ZThmZDBcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmVzcG9uc2UtY29udGVudC10eXBlPSomdXN

In [2]:
!wget https://huggingface.co/datasets/Sakhiur/signal/resolve/main/X_test_Sakhiur.csv

--2026-08-25 03:45:21--  https://huggingface.co/datasets/Sakhiur/signal/resolve/main/X_test_Sakhiur.csv
Resolving huggingface.co (huggingface.co)... 18.65.14.100, 18.65.14.125, 18.65.14.85, ...
Connecting to huggingface.co (huggingface.co)|18.65.14.100|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/datasets/Sakhiur/signal/d35627bf94d1a8ad170c5553b07812318821ee95/X_test_Sakhiur.csv?%2Fdatasets%2FSakhiur%2Fsignal%2Fresolve%2Fmain%2FX_test_Sakhiur.csv=&etag=%2249ac492745f5c946abd59d29947ad5baa2a6d39d%22 [following]
--2026-08-25 03:45:21--  https://huggingface.co/api/resolve-cache/datasets/Sakhiur/signal/d35627bf94d1a8ad170c5553b07812318821ee95/X_test_Sakhiur.csv?%2Fdatasets%2FSakhiur%2Fsignal%2Fresolve%2Fmain%2FX_test_Sakhiur.csv=&etag=%2249ac492745f5c946abd59d29947ad5baa2a6d39d%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 10184720 (9.7M) [text/plain]
Saving to: 

In [3]:
!wget https://huggingface.co/datasets/Sakhiur/signal/resolve/main/y_train_resampled_Sakhiur.csv

--2026-08-25 03:45:22--  https://huggingface.co/datasets/Sakhiur/signal/resolve/main/y_train_resampled_Sakhiur.csv
Resolving huggingface.co (huggingface.co)... 18.65.14.100, 18.65.14.125, 18.65.14.85, ...
Connecting to huggingface.co (huggingface.co)|18.65.14.100|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/datasets/Sakhiur/signal/d35627bf94d1a8ad170c5553b07812318821ee95/y_train_resampled_Sakhiur.csv?%2Fdatasets%2FSakhiur%2Fsignal%2Fresolve%2Fmain%2Fy_train_resampled_Sakhiur.csv=&etag=%22d1dadfe2ca86451459223e55ad3fb4a057c0f083%22 [following]
--2026-08-25 03:45:22--  https://huggingface.co/api/resolve-cache/datasets/Sakhiur/signal/d35627bf94d1a8ad170c5553b07812318821ee95/y_train_resampled_Sakhiur.csv?%2Fdatasets%2FSakhiur%2Fsignal%2Fresolve%2Fmain%2Fy_train_resampled_Sakhiur.csv=&etag=%22d1dadfe2ca86451459223e55ad3fb4a057c0f083%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 

In [4]:
!wget https://huggingface.co/datasets/Sakhiur/signal/resolve/main/y_test_Sakhiur.csv

--2026-08-25 03:45:23--  https://huggingface.co/datasets/Sakhiur/signal/resolve/main/y_test_Sakhiur.csv
Resolving huggingface.co (huggingface.co)... 18.65.14.100, 18.65.14.125, 18.65.14.85, ...
Connecting to huggingface.co (huggingface.co)|18.65.14.100|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /api/resolve-cache/datasets/Sakhiur/signal/d35627bf94d1a8ad170c5553b07812318821ee95/y_test_Sakhiur.csv?%2Fdatasets%2FSakhiur%2Fsignal%2Fresolve%2Fmain%2Fy_test_Sakhiur.csv=&etag=%2265b12cabc0360608ebbdad6a81d0164f0522455e%22 [following]
--2026-08-25 03:45:24--  https://huggingface.co/api/resolve-cache/datasets/Sakhiur/signal/d35627bf94d1a8ad170c5553b07812318821ee95/y_test_Sakhiur.csv?%2Fdatasets%2FSakhiur%2Fsignal%2Fresolve%2Fmain%2Fy_test_Sakhiur.csv=&etag=%2265b12cabc0360608ebbdad6a81d0164f0522455e%22
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 200 OK
Length: 1199517 (1.1M) [text/plain]
Saving to: ‘

In [6]:
import pandas as pd
import numpy as np
import random
import joblib
import os

np.random.seed(42)
random.seed(42)

X_train = pd.read_csv('X_train_resampled_Sakhiur.csv')
y_train = pd.read_csv('y_train_resampled_Sakhiur.csv').iloc[:, 0]
X_test = pd.read_csv('X_test_Sakhiur.csv')
y_test = pd.read_csv('y_test_Sakhiur.csv').iloc[:, 0]

print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (1049480, 25) Test: (150325, 25)


## 2. Hyperparameter search space
Same structure as `SOTA_Paper_8.ipynb` cell 39, trimmed to the four models your project actually uses (decision tree, RF, XGBoost, logistic regression) and depth-capped to stay inside Colab free-tier limits, per the compute constraint already flagged for this project.

In [7]:
param_distributions = {
    'DecisionTreeClassifier': {
        'max_depth': [5, 10, 15, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'RandomForestClassifier': {
        'n_estimators': [100, 150, 200],
        'max_depth': [10],  # capped per project compute constraint, do not widen
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'XGBClassifier': {
        'n_estimators': [100, 150, 200],
        'max_depth': [6, 8, 10],
        'learning_rate': [0.01, 0.05, 0.1]
    },
    'LogisticRegression': {
        'C': [0.01, 0.1, 1.0, 10.0],
        'max_iter': [1000]
    }
}

## 3. Randomized search per model

In [8]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

base_models = {
    'DecisionTreeClassifier': DecisionTreeClassifier(random_state=42),
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'XGBClassifier': XGBClassifier(random_state=42, eval_metric='mlogloss'),
    'LogisticRegression': LogisticRegression(random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_params = {}
best_estimators = {}

for name, model in base_models.items():
    print(f'Searching {name}...')
    search = RandomizedSearchCV(
        model, param_distributions[name],
        n_iter=10, cv=cv, scoring='f1_macro',
        random_state=42, n_jobs=-1
    )
    search.fit(X_train, y_train)
    best_params[name] = search.best_params_
    best_estimators[name] = search.best_estimator_
    print(f'  Best params: {search.best_params_}')
    print(f'  Best CV macro F1: {search.best_score_:.4f}')

Searching DecisionTreeClassifier...
  Best params: {'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 20}
  Best CV macro F1: 0.7718
Searching RandomForestClassifier...
  Best params: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 10}
  Best CV macro F1: 0.7218
Searching XGBClassifier...


ValueError: 
All the 50 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
50 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/xgboost/core.py", line 569, in inner_f
    return func(**kwargs)
  File "/usr/local/lib/python3.13/dist-packages/xgboost/sklearn.py", line 1812, in fit
    raise ValueError(
    ...<2 lines>...
    )
ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2 3], got ['Lower' 'Middle' 'No_income' 'Upper']


## 4. Evaluate each tuned model on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

test_results = {}
for name, model in best_estimators.items():
    y_pred = model.predict(X_test)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    test_results[name] = macro_f1
    print(f'--- {name} ---')
    print(classification_report(y_test, y_pred, digits=3))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {name}')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.show()

## 5. Ensembling: hard + soft voting
Adapted from `SOTA_Paper_8.ipynb` cell 44. `voting='soft'` requires every base estimator to support `predict_proba`, which all four models here do.

In [ ]:
from sklearn.ensemble import VotingClassifier

voting_clf_hard = VotingClassifier(
    estimators=[(name, model) for name, model in best_estimators.items()],
    voting='hard'
)
voting_clf_soft = VotingClassifier(
    estimators=[(name, model) for name, model in best_estimators.items()],
    voting='soft'
)

print('Training hard voting ensemble...')
voting_clf_hard.fit(X_train, y_train)
print('Training soft voting ensemble...')
voting_clf_soft.fit(X_train, y_train)

## 6. Evaluate ensembles

In [ ]:
for clf, label in zip([voting_clf_hard, voting_clf_soft], ['Hard Voting', 'Soft Voting']):
    y_pred = clf.predict(X_test)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    test_results[label] = macro_f1
    print(f'--- {label} ---')
    print(classification_report(y_test, y_pred, digits=3))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {label}')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.show()

print('\nFinal comparison (macro F1):')
for name, score in sorted(test_results.items(), key=lambda x: -x[1]):
    print(f'{name}: {score:.4f}')

## 7. Save models and metrics
This is the handoff point for SPR-04 (figures) and SPR-05 (SHAP), both of which read these saved models read-only and must not retrain them.

In [ ]:
os.makedirs('models/decision_tree', exist_ok=True)
os.makedirs('models/random_forest', exist_ok=True)
os.makedirs('models/xgboost', exist_ok=True)
os.makedirs('models/logistic_regression', exist_ok=True)
os.makedirs('results/tables', exist_ok=True)

joblib.dump(best_estimators['DecisionTreeClassifier'], 'models/decision_tree/model.joblib')
joblib.dump(best_estimators['RandomForestClassifier'], 'models/random_forest/model.joblib')
joblib.dump(best_estimators['XGBClassifier'], 'models/xgboost/model.joblib')
joblib.dump(best_estimators['LogisticRegression'], 'models/logistic_regression/model.joblib')
joblib.dump(voting_clf_soft, 'models/voting_ensemble_soft.joblib')
joblib.dump(voting_clf_hard, 'models/voting_ensemble_hard.joblib')

results_df = pd.DataFrame(list(test_results.items()), columns=['model', 'macro_f1'])
results_df = results_df.sort_values('macro_f1', ascending=False)
results_df.to_csv('results/tables/model_comparison.csv', index=False)
print(results_df)
print('\nBest model for SPR-05 SHAP analysis:', results_df.iloc[0]['model'])